# Week 2b: YOLO11n at 640 px, and two more seeds at 960 px

Three training runs in one session, one after the other:

1. YOLO11n at 640 px (seed 0). RT-DETR-l was trained at 640 px and YOLO11n at 960 px, so this run separates the effect of the model from the effect of the input size.
2. and 3. YOLO11n at 960 px with seeds 1 and 2. Together with the week 1 run (seed 0) this shows how much the scores move between two normal training runs.

Before running: Accelerator = GPU T4 x2 (or P100), Internet = on, add the LOCO image dataset with + Add Input. Then Save Version > **Save & Run All (Commit)**, not Quick Save.

In [ ]:
!pip install -q ultralytics

In [ ]:
# Annotations (COCO format) from the LOCO repository
!git clone -q --depth 1 https://github.com/tum-fml/loco /kaggle/working/loco
!ls -la /kaggle/working/loco/rgb

In [ ]:
%%writefile /kaggle/working/coco_to_yolo.py
"""Convert the LOCO annotations (COCO format) to the YOLO format.

Usage:
    python scripts/coco_to_yolo.py --ann-dir path/to/loco/rgb \
        --images-root path/to/downloaded/images --out data/loco_yolo

Result:
    data/loco_yolo/images/train/*.jpg   (links or copies of the original images)
    data/loco_yolo/labels/train/*.txt   (one line per box: class x_center y_center w h, 0..1)
    data/loco_yolo/images/val/..., data/loco_yolo/labels/val/...
    data/loco_yolo/loco.yaml            (dataset file for Ultralytics)

The images are found by file name anywhere below --images-root, so the
folder layout of the download does not matter.
"""

import argparse
import json
import os
import shutil
from pathlib import Path

SUBSETS = {
    "loco-sub1-v1-val.json": "val",
    "loco-sub2-v1-train.json": "train",
    "loco-sub3-v1-train.json": "train",
    "loco-sub4-v1-val.json": "val",
    "loco-sub5-v1-train.json": "train",
}

# LOCO category id -> YOLO class index
CLASSES = {3: "small_load_carrier", 5: "forklift", 7: "pallet", 10: "stillage", 11: "pallet_truck"}
INDEX = {cat_id: i for i, cat_id in enumerate(CLASSES)}


def yolo_line(bbox, width, height, cls):
    """COCO box [x, y, w, h] in pixels -> YOLO line, clipped to the image."""
    x, y, w, h = bbox
    x0, y0 = max(x, 0.0), max(y, 0.0)
    x1, y1 = min(x + w, width), min(y + h, height)
    if x1 <= x0 or y1 <= y0:
        return None
    xc = (x0 + x1) / 2 / width
    yc = (y0 + y1) / 2 / height
    return f"{cls} {xc:.6f} {yc:.6f} {(x1 - x0) / width:.6f} {(y1 - y0) / height:.6f}"


def place_image(src, dst, copy):
    if dst.exists() or dst.is_symlink():
        return
    if not copy:
        try:
            os.symlink(src.resolve(), dst)
            return
        except OSError:  # e.g. Windows without symlink rights
            pass
    shutil.copy2(src, dst)


def convert(ann_dir, images_root, out, copy=False):
    index = {}
    if images_root is not None:
        for p in images_root.rglob("*.jpg"):
            index[p.name] = p

    summary = {}
    for file_name, split in SUBSETS.items():
        data = json.loads((ann_dir / file_name).read_text())
        img_dir = out / "images" / split
        lbl_dir = out / "labels" / split
        img_dir.mkdir(parents=True, exist_ok=True)
        lbl_dir.mkdir(parents=True, exist_ok=True)

        lines = {im["id"]: [] for im in data["images"]}
        info = {im["id"]: im for im in data["images"]}
        skipped = 0
        for a in data["annotations"]:
            im = info[a["image_id"]]
            line = yolo_line(a["bbox"], im["width"], im["height"], INDEX[a["category_id"]])
            if line is None:
                skipped += 1
            else:
                lines[a["image_id"]].append(line)

        missing = 0
        for image_id, rows in lines.items():
            name = info[image_id]["file_name"]
            (lbl_dir / (Path(name).stem + ".txt")).write_text("\n".join(rows) + "\n")
            if images_root is not None:
                if name in index:
                    place_image(index[name], img_dir / name, copy)
                else:
                    missing += 1

        s = summary.setdefault(split, {"images": 0, "boxes": 0, "skipped": 0, "missing": 0})
        s["images"] += len(lines)
        s["boxes"] += sum(len(r) for r in lines.values())
        s["skipped"] += skipped
        s["missing"] += missing

    names = "\n".join(f"  {i}: {n}" for i, n in enumerate(CLASSES.values()))
    (out / "loco.yaml").write_text(
        f"path: {out.resolve()}\ntrain: images/train\nval: images/val\nnames:\n{names}\n")
    return summary


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--ann-dir", type=Path, required=True, help="the rgb/ folder of the LOCO repository")
    parser.add_argument("--images-root", type=Path, help="folder with the downloaded LOCO images")
    parser.add_argument("--out", type=Path, default=Path("data/loco_yolo"))
    parser.add_argument("--copy", action="store_true", help="copy images instead of linking them")
    args = parser.parse_args()

    summary = convert(args.ann_dir, args.images_root, args.out, args.copy)
    for split, s in summary.items():
        print(f"{split}: {s['images']} images, {s['boxes']} boxes, "
              f"{s['skipped']} empty boxes skipped, {s['missing']} images not found")
    print(f"dataset file: {args.out / 'loco.yaml'}")


if __name__ == "__main__":
    main()


In [ ]:
import sys, subprocess, zipfile, tarfile
from pathlib import Path

# If the uploaded dataset still contains archives (for example one zip per subset),
# unpack them first. Plain .jpg files are used where they are.
extract_dir = Path("/kaggle/working/loco_images")
archives = [p for p in Path("/kaggle/input").rglob("*") if p.suffix.lower() in {".zip", ".tar", ".gz", ".tgz"}]
for a in archives:
    print("unpacking", a)
    target = extract_dir / a.stem
    target.mkdir(parents=True, exist_ok=True)
    if a.suffix.lower() == ".zip":
        zipfile.ZipFile(a).extractall(target)
    else:
        tarfile.open(a).extractall(target)

n_input = sum(1 for _ in Path("/kaggle/input").rglob("*.jpg"))
n_extracted = sum(1 for _ in extract_dir.rglob("*.jpg")) if extract_dir.exists() else 0
images_root = "/kaggle/working/loco_images" if n_extracted > n_input else "/kaggle/input"
print(f"jpg files: {n_input} in /kaggle/input, {n_extracted} unpacked -> using {images_root}")

# The images are found by file name anywhere below images_root,
# so it does not matter how the uploaded dataset is organised.
out = subprocess.run([sys.executable, "/kaggle/working/coco_to_yolo.py",
                      "--ann-dir", "/kaggle/working/loco/rgb",
                      "--images-root", images_root,
                      "--out", "/kaggle/working/loco_yolo"],
                     capture_output=True, text=True)
print(out.stdout, out.stderr)
assert out.stdout.count(" 0 images not found") == 2, "Some images were not found: check the uploaded dataset"
print(open("/kaggle/working/loco_yolo/loco.yaml").read())

In [ ]:
import csv, os, shutil
from collections import Counter
from pathlib import Path
from ultralytics import YOLO

DATA = "/kaggle/working/loco_yolo/loco.yaml"
OUT = Path("/kaggle/working/out")
OUT.mkdir(exist_ok=True)

# boxes per class in val, counted from the label files
counts = Counter()
lbl_dir = "/kaggle/working/loco_yolo/labels/val"
for f in os.listdir(lbl_dir):
    for line in open(os.path.join(lbl_dir, f)):
        if line.strip():
            counts[int(line.split()[0])] += 1

def run(imgsz, seed, keep_weights):
    name = f"yolo11n_{imgsz}_seed{seed}"
    model = YOLO("yolo11n.pt")
    model.train(data=DATA, imgsz=imgsz, epochs=100, patience=30, batch=16, max_det=500,
                seed=seed, deterministic=True, workers=4, time=3,
                project="/kaggle/working/runs", name=name, exist_ok=True)
    train_dir = Path(model.trainer.save_dir)
    m = YOLO(train_dir / "weights" / "best.pt").val(
        data=DATA, imgsz=imgsz, batch=16, max_det=500, conf=0.001, split="val", plots=True,
        project="/kaggle/working/runs", name=name + "_val", exist_ok=True)

    rows = []
    for i, c in enumerate(m.box.ap_class_index):
        p, r, ap50, ap = m.box.class_result(i)
        rows.append([m.names[int(c)], counts[int(c)], round(float(p), 3), round(float(r), 3), round(float(ap50), 3), round(float(ap), 3)])
    rows.append(["all", sum(counts.values()), round(float(m.box.mp), 3), round(float(m.box.mr), 3), round(float(m.box.map50), 3), round(float(m.box.map), 3)])
    with open(OUT / f"{name}.csv", "w", newline="") as f:
        w = csv.writer(f)
        w.writerow(["class", "val_boxes", "precision", "recall", "AP50", "AP50-95"])
        w.writerows(rows)
    shutil.copy(train_dir / "results.csv", OUT / f"{name}_train_log.csv")
    cm = Path(m.save_dir) / "confusion_matrix_normalized.png"
    if cm.exists():
        shutil.copy(cm, OUT / f"{name}_confusion_matrix_normalized.png")
    if keep_weights:
        shutil.copy(train_dir / "weights" / "best.pt", OUT / f"{name}_best.pt")
    print(name, "mAP50-95", round(float(m.box.map), 3))

for imgsz, seed, keep in [(640, 0, True), (960, 1, False), (960, 2, False)]:
    run(imgsz, seed, keep)

shutil.rmtree("/kaggle/working/loco_yolo", ignore_errors=True)
shutil.rmtree("/kaggle/working/loco", ignore_errors=True)
shutil.rmtree("/kaggle/working/loco_images", ignore_errors=True)
print(sorted(os.listdir(OUT)))